# トヨタ自動車 未来創成センター 論文一覧スクレイピング

今井先生の元タスク文書「Build a Researcher Database」の1番目の対象。
公式検索ページ(https://www.toyota.co.jp/jpn/tech/partner_robot/search/)はJS描画だが、
裏で叩いているJSON APIを直接取得すれば構造化データがそのまま手に入ることを確認済み。

- データ源: `https://www.toyota.co.jp/jpn/tech/partner_robot/search/json/articles.json`
- robots.txt確認済み: `/jpn/tech/partner_robot/`は`Disallow`に含まれない
- 件数: 154件（2018〜2026年）
- 著者名は`*Name*`のようにアスタリスクで囲まれた部分がトヨタ所属著者を表す（サイト自身のJS `search.js` の`formatAuthors`関数と同じ正規表現`\*([^\*]+)\*`で抽出できることを確認済み。154件全件で崩れ・奇数アスタリスクなし）。
- 次のステップ（このノートブックの範囲外）: タイトルベースでOpenAlex IDに紐付け、指標計算に使う。

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import re

import pandas as pd
import requests

# 実行場所に依存せず、リポジトリ直下を探索する。
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'AGENTS.md').exists() and (candidate / 'data').is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError('リポジトリ直下を特定できません。リポジトリ内で実行してください。')

OUTPUT_DIR = ROOT / 'data' / 'derived' / 'toyota_official_scrape'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_URL = 'https://www.toyota.co.jp/jpn/tech/partner_robot/search/json/articles.json'
USER_AGENT = 'Mozilla/5.0 (compatible; Sci-sci-research-bot/1.0; contact: r.ikura@msp-lab.org)'

CATEGORY_MAP = {
    0: '全て',
    1: 'バイオヒューマン',
    2: 'ロボティクス',
    3: 'エネルギ',
    4: '数理AI',
    5: '革新的構造',
}

RUN_AT_UTC = datetime.now(timezone.utc).isoformat()
print(f'RUN_AT_UTC = {RUN_AT_UTC}')

RUN_AT_UTC = 2026-07-27T02:55:18.294084+00:00


## 1. 取得

サイト自身のフロントエンド(`search.js`)が使っているのと同じJSONエンドポイントを直接取得する。
検索フォームの操作や大量ページ遷移は行わず、1回のGETで全件を取得する。

In [2]:
resp = requests.get(SOURCE_URL, headers={'User-Agent': USER_AGENT, 'Accept': 'application/json'}, timeout=30)
resp.raise_for_status()
articles = resp.json()
print(f'取得件数: {len(articles)}')
print(f'HTTPステータス: {resp.status_code}')
articles[:2]

取得件数: 154
HTTPステータス: 200


[{'category': 2,
  'recommend': None,
  'note': None,
  'authors': 'Obata, M., *Iwanaga, Y.*, *Tsuchinaga, M.*, Mizutani, A., Downes, D., & Lee, S.',
  'title': 'Towards the Development of a Robot for Well-Being: Applying the Motivation, Engagement, and Thriving in User Experience (METUX) Model',
  'source': 'The 2026 ACM/IEEE International Conference on Human-Robot Interaction (HRI2026) LBR',
  'published_at': '2026/03/16',
  'year': 2026,
  'link': 'https://dl.acm.org/doi/10.1145/3776734.3794360'},
 {'category': 4,
  'recommend': 1,
  'note': 'Full Paper Acceptance Rate 21.9%',
  'authors': 'Sone, Y., Ninomiya, Y., Miwa, K., *Sumi, Y.*, *Nakanishi, R.*, *Mitsuda, E.*, *Sato, K.* & *Odashima, T.*',
  'title': 'Surface vs. Structural Similarity: Aligning Computational Metrics with Human Judgments',
  'source': '21st The International Conference on Computer Graphics, Interaction, Visualization Theory and Applications',
  'published_at': '2026/03/09',
  'year': 2026,
  'link': '#'}]

## 2. 構造化（トヨタ所属著者の抽出）

`*Name*`のアスタリスク部分がトヨタ所属著者。サイト自身のJS(`formatAuthors`)と同じ正規表現`\*([^\*]+)\*`で抽出する。
著者欄全体を個々の著者に厳密分割することはしない（表記ゆれが大きく、この段階では不要。太字＝トヨタ所属の抽出だけで今井先生の要件を満たす）。

In [3]:
BOLD_PATTERN = re.compile(r'\*([^*]+)\*')


def extract_toyota_authors(authors_raw):
    """著者欄からアスタリスクで囲まれたトヨタ所属著者名を抜き出す。"""
    return [m.strip() for m in BOLD_PATTERN.findall(authors_raw)]


def strip_markup(authors_raw):
    """表示用に、著者欄からアスタリスクだけを除いた素のテキストを作る。"""
    return authors_raw.replace('*', '')


rows = []
for a in articles:
    toyota_authors = extract_toyota_authors(a['authors'])
    rows.append({
        'title': a['title'],
        'authors_raw': a['authors'],
        'authors_display': strip_markup(a['authors']),
        'toyota_affiliated_authors': '; '.join(toyota_authors),
        'n_toyota_affiliated_authors': len(toyota_authors),
        'category_id': a['category'],
        'category_name': CATEGORY_MAP.get(a['category'], f"unknown({a['category']})"),
        'source': a['source'],
        'year': a['year'],
        'published_at': a['published_at'],
        'link': a['link'],
        'note': a.get('note'),
        'recommend': a.get('recommend'),
    })

df = pd.DataFrame(rows)
print(f'行数: {len(df)}')
print(f'太字著者が0人の行: {(df["n_toyota_affiliated_authors"] == 0).sum()}')
print(f'リンクなし(#)の行: {(df["link"] == "#").sum()}')
df.head(5)

行数: 154
太字著者が0人の行: 0
リンクなし(#)の行: 4


,title,authors_raw,authors_display,toyota_affiliated_authors,n_toyota_affiliated_authors,category_id,category_name,source,year,published_at,link,note,recommend
0,Towards the Development of a Robot for Well-Be...,"Obata, M., *Iwanaga, Y.*, *Tsuchinaga, M.*, Mi...","Obata, M., Iwanaga, Y., Tsuchinaga, M., Mizuta...","Iwanaga, Y.; Tsuchinaga, M.",2,2,ロボティクス,The 2026 ACM/IEEE International Conference on ...,2026,2026/03/16,https://dl.acm.org/doi/10.1145/3776734.3794360,NaN,None
1,Surface vs. Structural Similarity: Aligning Co...,"Sone, Y., Ninomiya, Y., Miwa, K., *Sumi, Y.*, ...","Sone, Y., Ninomiya, Y., Miwa, K., Sumi, Y., Na...","Sumi, Y.; Nakanishi, R.; Mitsuda, E.; Sato, K....",5,4,数理AI,21st The International Conference on Computer ...,2026,2026/03/09,#,Full Paper Acceptance Rate 21.9%,1
2,VDoF-MPC: Variable Degrees-of-Freedom Whole-Bo...,"Jin, T., Kobayashi, T., & *Doi, M*.","Jin, T., Kobayashi, T., & Doi, M.","Doi, M",1,2,ロボティクス,IEEE Access,2026,2026/01/21,https://ieeexplore.ieee.org/document/11359690,NaN,None
3,A Graph-Based Heuristic for Bus Route Planning...,"Yu, Z., Lim, B. G. S., Tan, R. R. P., *Kaji, H...","Yu, Z., Lim, B. G. S., Tan, R. R. P., Kaji, H....","Kaji, H.",1,4,数理AI,IEEE Access,2026,2026/01/12,https://ieeexplore.ieee.org/abstract/document/...,NaN,None
4,Jump Amplification of a Legged Jumping Robot w...,"Itsuno, T., Takakuwa, R., Ito, F., *Doi, M.*, ...","Itsuno, T., Takakuwa, R., Ito, F., Doi, M., Ko...","Doi, M.; Kondo, H.",2,2,ロボティクス,SII2026,2026,2026/01/11,#,NaN,None


## 3. 保存

生JSON(取得日時つき、再現性のため)と、構造化済みCSVの両方を保存する。

In [4]:
raw_json_path = OUTPUT_DIR / 'frontier_research_center_raw.json'
with raw_json_path.open('w', encoding='utf-8') as f:
    json.dump({'run_at_utc': RUN_AT_UTC, 'source_url': SOURCE_URL, 'articles': articles}, f, ensure_ascii=False, indent=2)

csv_path = OUTPUT_DIR / 'frontier_research_center.csv'
df.to_csv(csv_path, index=False)

print(f'保存: {raw_json_path.relative_to(ROOT)}')
print(f'保存: {csv_path.relative_to(ROOT)}')

保存: data/derived/toyota_official_scrape/frontier_research_center_raw.json
保存: data/derived/toyota_official_scrape/frontier_research_center.csv


## 4. ざっと確認

年別・カテゴリ別の分布、トヨタ所属著者の頻出上位を見て、データの妥当性を確認する。

In [5]:
print('年別件数:')
print(df['year'].value_counts().sort_index())
print()
print('カテゴリ別件数:')
print(df['category_name'].value_counts())

年別件数:
year
2018    26
2019    17
2020    18
2021    13
2022    11
2023    14
2024    16
2025    34
2026     5
Name: count, dtype: int64

カテゴリ別件数:
category_name
ロボティクス      57
数理AI        46
バイオヒューマン    24
エネルギ        23
革新的構造        4
Name: count, dtype: int64


In [6]:
from collections import Counter

all_toyota_authors = Counter()
for names in df['toyota_affiliated_authors']:
    for n in names.split('; '):
        if n:
            all_toyota_authors[n] += 1

pd.DataFrame(all_toyota_authors.most_common(20), columns=['name', 'n_papers'])

,name,n_papers
0,"Kaji, H.",19
1,"Masuda, T.",14
2,"Takeshita, K.",12
3,"Yamamoto, T.",11
4,"Jimbo, T.",8
5,"Ito, M.",7
6,"Takahashi, T.",7
7,"Kanai, T.",6
8,"Odashima, T.",5
9,"Doi, M.",5
